In [2]:
!pip install transformers

Defaulting to user installation because normal site-packages is not writeable


In [3]:
%pip install accelerate

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install pydantic

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
import json
import re
from typing import Optional
from pydantic import BaseModel, Field, ValidationError


# Define your expected output schema (customize fields to your task)
class ExtractionSchema(BaseModel):
  date: Optional[str] = Field(
      default=None, description="Extracted date or null"
  )
  location: Optional[str] = Field(
      default=None, description="Extracted location or null"
  )
  summary: Optional[str] = Field(
      default=None, description="Brief summary or null"
  )


def parse_and_validate_output(raw_output: str) -> dict:
  """Safely extracts JSON from raw LLM output and validates it against Pydantic."""
  json_str = None

  # 1. Try direct parsing
  try:
    data = json.loads(raw_output)
    if isinstance(data, dict):
      json_str = data
  except json.JSONDecodeError:
    pass

  # 2. Fallback: Search for JSON block via regex if direct parsing failed
  if not json_str:
    match = re.search(r"(\{.*?\})", raw_output, re.DOTALL)
    if match:
      try:
        data = json.loads(match.group(1))
        if isinstance(data, dict):
          json_str = data
      except json.JSONDecodeError:
        pass

  # 3. If we found a dictionary, validate with Pydantic
  if json_str:
    try:
      validated = ExtractionSchema(**json_str)
      return validated.dict(), True  # Success
    except ValidationError:
      # Schema mismatch, but we got JSON
      return json_str, False

  # 4. Total failure fallback
  return {"date": None, "location": None, "summary": None}, False

# 07 - Metadata extraction with EuroLLM

Run from the `notebooks/` folder. Install once: `%pip install transformers accelerate torch`.

Steps: (1) make a gold-standard template you verify by hand, (2) read the first pages of each PDF, (3) ask EuroLLM for JSON, (4) check the answers against the page text and against your gold rows.

Everything runs locally. Outputs go to `../output/extraction/`.

## 1. Settings

In [6]:
from pathlib import Path
import json, re, statistics, unicodedata, difflib
import pandas as pd
import pymupdf
from IPython.display import display

PDF_DIR = Path('../pdfs').resolve()
INVENTORY_DIR = Path('../output/inventory').resolve()
OUT_DIR = Path('../output/extraction').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)


MODEL_ID = MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
RUN_SCOPE = 'dev'        
FIRST_PAGES = 3              
MAX_INPUT_TOKENS = 3000      
FIELDS = ['title', 'authors', 'doi', 'publication_year', 'document_type', 'language', 'keywords']
LIST_FIELDS = {'authors', 'keywords'}
DOC_TYPES = ['journal_article', 'report', 'thesis', 'conference_paper', 'book_chapter', 'policy_brief', 'other']

read = lambda name: pd.read_csv(INVENTORY_DIR / name, dtype=str, keep_default_na=False, encoding='utf-8-sig')
documents, observations = read('documents.csv'), read('metadata_observations.csv')
print(len(documents), 'documents')

93 documents


## 2. Gold-standard template

Creates `gold_metadata.csv` once (never overwrites your edits): 40 documents, 25 `dev` and 15 `test`, pre-filled with the PDF-embedded candidates. **Verify every value against the PDF or the Publinova record.**

Fill-in rules: `;` between authors/keywords · `publication_year` as 4 digits · `document_type` from `journal_article, report, thesis, conference_paper, book_chapter, policy_brief, other` · `language` is `en`, `nl` or `mixed` · empty = not checked · `NONE` = checked and absent. Don't tune on `test` until the end.

In [7]:
GOLD_PATH = OUT_DIR / 'gold_metadata.csv'
if GOLD_PATH.exists():
    print('gold_metadata.csv exists; left untouched.')
else:
    pool = documents[documents.text_status.isin(['extractable', 'partial'])]
    pick = pool.sample(n=min(40, len(pool)), random_state=42)
    cand = lambda d, f: ' | '.join(observations[(observations.document_id == d) & (observations.field_name == f)].observed_value.unique())
    gold = pd.DataFrame({
        'document_id': pick.document_id, 'file_name': pick.file_name,
        'split': ['dev'] * min(25, len(pick)) + ['test'] * max(len(pick) - 25, 0),
        'title': [cand(d, 'title') for d in pick.document_id],
        'authors': [cand(d, 'author') for d in pick.document_id],
        'doi': [cand(d, 'doi') for d in pick.document_id],
        'publication_year': '', 'document_type': '', 'language': '',
        'keywords': [cand(d, 'pdf_keywords') for d in pick.document_id]})
    gold.to_csv(GOLD_PATH, index=False, encoding='utf-8-sig')
    print('Created', GOLD_PATH, '- now verify it by hand.')

gold_metadata.csv exists; left untouched.


## 3. Input and prompt

Each line of the first pages is read with its font size. Lines clearly larger than the page's typical text get a `[BIG]` marker, which helps the model find the title. The prompt ends with `{` so the model continues straight into a JSON object.

In [8]:
def page_text(page):
    lines = []
    for block in page.get_text('dict', sort=True)['blocks']:
        if block.get('type') != 0:
            continue
        for line in block['lines']:
            spans = [s for s in line['spans'] if s['text'].strip()]
            if spans:
                lines.append((max(s['size'] for s in spans), ''.join(s['text'] for s in spans).strip()))
    if not lines:
        return ''
    typical = statistics.median(size for size, _ in lines)
    return '\n'.join(('[BIG] ' if size >= 1.2 * typical else '') + text for size, text in lines)

def first_relevant_pages(pdf_path, min_chars=1200, max_pages=FIRST_PAGES):
    with pymupdf.open(PDF_DIR / pdf_path) as pdf:
        texts = []
        total = 0
        for i in range(min(max_pages + 1, len(pdf))):
            t = page_text(pdf[i])
            texts.append(f'--- page {i + 1} ---\n{t}')
            total += len(t.strip())
            if total >= min_chars:
                break
        return '\n\n'.join(texts)

SYSTEM = (
    'You are a bibliographic metadata extractor. '
    'You read the first page of a document (English or Dutch) and return its metadata as JSON. '
    'Use ONLY information printed in the document. If a field is not printed, use null or []. '
    'Lines starting with [BIG] are in a larger font and are usually the title. '
    'Respond with ONE JSON object and NOTHING else. '
    'Do not repeat these instructions. Do not add any text before or after the JSON. '
    'The JSON must have exactly these keys: '
    'title (string or null), '
    'authors (list of strings), '
    'doi (string or null, without "https://doi.org/" prefix), '
    'publication_year (integer or null, from "Published" or the journal issue year), '
    'document_type (one of: ' + ', '.join(DOC_TYPES) + '), '
    'language ("en", "nl", "mixed", or null), '
    'keywords (list of strings). '
    'Example output for a made-up document:\n'
    '{"title": "The Example Study", "authors": ["A. Smith", "B. Jones"], '
    '"doi": "10.1234/example", "publication_year": 2024, '
    '"document_type": "journal_article", "language": "en", '
    '"keywords": ["alpha", "beta"]}'
)

def user_prompt(text):
    return (
        'Extract the metadata from the document below and return it as a single JSON object.\n\n'
        'DOCUMENT:\n'
        f'{text}'
    )

demo = documents[documents.text_status != 'no_text'].iloc[0]
print(demo.file_name); print(first_relevant_pages(demo.pdf_path)[:1500])

001_adaptive-capacities-for-diversified-flood-risk-management-strategies-learning-from-pilot-projects.pdf
--- page 1 ---
[BIG] water
Article
[BIG] Adaptive Capacities for Diversiﬁed Flood Risk
[BIG] Management Strategies:Learning from Pilot Projects
Flavia Simona Cosoveanu *, Jean-Marie Buijs *, Marloes Bakker and Teun Terpstra
Department of Technology, Water & Environment; HZ University of Applied Sciences, Het Groene Woud 1,
4331 NB Middelburg, The Netherlands; m.h.n.bakker@hz.nl (M.B.); t.terpstra@hz.nl (T.T.)
*Correspondence:coso0002@hz.nl (F.S.C.); jm.buijs@hz.nl (J.-M.B.)


Received:17 October 2019; Accepted:10 December 2019; Published:14 December 2019
Abstract:Diversiﬁcation of ﬂood risk management strategies (FRMS) in response to climate change
reliesontheadaptivecapacitiesofinstitutions.Althoughadaptivecapacitiesenableﬂexibility
and adjustment, more empirical research is needed to better grasp the role of adaptive capacities
toaccommodateexpectedclimatechangeeﬀec

## 4. Load EuroLLM and run

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
model = model.to("cpu")
model.eval()

def extract(text):
    messages = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user_prompt(text)}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(
    prompt,
    return_tensors='pt',
    truncation=True,
    max_length=MAX_INPUT_TOKENS,
    add_special_tokens=False,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id,
        )
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

/var/data/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 5636.83it/s]


In [10]:
RAW_PATH = OUT_DIR / 'eurollm_raw.jsonl'
runnable = documents[documents.text_status.isin(['extractable', 'partial'])]
if RUN_SCOPE == 'sample5':
    todo = runnable.head(5)
elif RUN_SCOPE == 'all':
    todo = runnable
else:
    g = pd.read_csv(GOLD_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    todo = runnable[runnable.document_id.isin(g.loc[g.split == RUN_SCOPE, 'document_id'])]

done = {}
if RAW_PATH.exists():
    for line in RAW_PATH.read_text(encoding='utf-8').splitlines():
        r = json.loads(line); done[(r['document_id'], r['model'])] = r['raw']

with RAW_PATH.open('a', encoding='utf-8') as sink:
    for n, row in enumerate(todo.itertuples(index=False), start=1):
        if (row.document_id, MODEL_ID) in done:
            continue
        raw = extract(first_relevant_pages(row.pdf_path))
        done[(row.document_id, MODEL_ID)] = raw
        sink.write(json.dumps({'document_id': row.document_id, 'model': MODEL_ID, 'raw': raw}, ensure_ascii=False) + '\n')
        sink.flush()
        print(f'[{n}/{len(todo)}] {row.file_name[:70]}')

[15/25] 045_circular-tourism-and-support-from-local-authorities.pdf
[16/25] 050_embedding-of-project-findings-in-educational-curricula.pdf
[17/25] 057_kijk-op-later-pilot.pdf
[18/25] 064_management-of-harmful-animal-activities-on-levees.pdf
[19/25] 067_monitor-wind-in-de-zeilen.pdf
[20/25] 069_o2lab-project-bedrijfsleven-structureel-betrekken-bij-ondernemersc
[21/25] 070_onderzoek-circulaire-economie.pdf
[22/25] 074_overflow-tests-on-dutch-levees.pdf
[23/25] 080_projectplan-leernetwerk-sport-bewegen-en-gezondheid-in-zeeland.pdf


KeyboardInterrupt: 

## 5. Parse and sanity-check

Parsing never crashes on bad JSON; it records `parse_ok = False`. Titles, authors and DOIs are also checked against the page text: `grounded_* = False` means the model wrote something that is not actually on the first pages, which usually means a hallucination.

In [11]:
# --- Recovery cell: rebuild state without re-running Part 4 ---
from pathlib import Path
import json, re, unicodedata, difflib
import pandas as pd
import pymupdf
from IPython.display import display

PDF_DIR        = Path('../pdfs').resolve()
INVENTORY_DIR  = Path('../output/inventory').resolve()
OUT_DIR        = Path('../output/extraction').resolve()
GOLD_PATH      = OUT_DIR / 'gold_metadata.csv'
RAW_PATH       = OUT_DIR / 'eurollm_raw.jsonl'

MODEL_ID        = 'Qwen/Qwen2.5-1.5B-Instruct'
RUN_SCOPE       = 'dev'
FIELDS          = ['title','authors','doi','publication_year','document_type','language','keywords']
LIST_FIELDS     = {'authors','keywords'}
DOC_TYPES       = ['journal_article','report','thesis','conference_paper','book_chapter','policy_brief','other']

read = lambda name: pd.read_csv(INVENTORY_DIR / name, dtype=str, keep_default_na=False, encoding='utf-8-sig')
documents = read('documents.csv')

# Rebuild todo exactly as Part 4 would have
runnable = documents[documents.text_status.isin(['extractable','partial'])]
g = pd.read_csv(GOLD_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
todo = runnable[runnable.document_id.isin(g.loc[g.split == RUN_SCOPE, 'document_id'])]

# Reload raw LLM outputs saved by Part 4
done = {}
if RAW_PATH.exists():
    for line in RAW_PATH.read_text(encoding='utf-8').splitlines():
        r = json.loads(line)
        done[(r['document_id'], r['model'])] = r['raw']

print(f'documents: {len(documents)}')
print(f'todo (dev split): {len(todo)}')
print(f'done (raw outputs loaded): {len(done)}')

documents: 93
todo (dev split): 25
done (raw outputs loaded): 23


In [ ]:
# --- Part 5 (improved): parse LLM output + observation fallback ---

# Ensure observations are available even if Part 1 was skipped
observations = pd.read_csv(
    INVENTORY_DIR / 'metadata_observations.csv',
    dtype=str, keep_default_na=False, encoding='utf-8-sig'
)

def squash(s):
    s = unicodedata.normalize('NFKD', str(s)).lower()
    return re.sub(r'[^a-z0-9]+', '', ''.join(c for c in s if not unicodedata.combining(c)))

def clean_doi(v):
    return re.sub(r'(?i)^(https?://(dx\.)?doi\.org/|doi:\s*)', '', str(v).strip()).rstrip('.,;)').lower()

def as_list(v):
    if v is None or v == '': return []
    if isinstance(v, str): v = re.split(r'[;|]', v)
    return [str(x).strip() for x in v if str(x).strip()]

def find_dois_in_pdf(pdf_path):
    """Return every DOI across all pages, including line-broken and URL-prefixed."""
    with pymupdf.open(PDF_DIR / pdf_path) as pdf:
        text = '\n'.join(p.get_text() for p in pdf)

    text = re.sub(r'https?://(?:dx\.)?doi\.org/', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bdoi\s*:\s*', '', text, flags=re.IGNORECASE)

    joined = re.sub(r'\s+', '', text)

    pattern = r'10\.\d{4,9}/[-._;()/:A-Z0-9]*[A-Z0-9]'
    found = set()
    for m in re.finditer(pattern, text,   re.IGNORECASE):
        found.add(m.group(0).rstrip('.,;)'))
    for m in re.finditer(pattern, joined, re.IGNORECASE):
        found.add(m.group(0).rstrip('.,;)'))

    return sorted(d for d in found if len(d.split('/', 1)[-1]) >= 4)

# --- Observation lookup (one value per document per field) ---
FIELD_MAP_OBS = {'title':'title', 'author':'authors', 'doi':'doi', 'pdf_keywords':'keywords'}
obs = observations[observations.field_name.isin(FIELD_MAP_OBS)].copy()
obs['field'] = obs.field_name.map(FIELD_MAP_OBS)
obs['priority'] = obs.source_type.map({'pdf_text': 0, 'pdf_embedded': 1}).fillna(2)
obs = obs.sort_values(['document_id', 'field', 'priority'])
obs = obs.drop_duplicates(['document_id', 'field'], keep='first')
OBS_MAP = {(r.document_id, r.field): r.observed_value for r in obs.itertuples()}

def is_valid(v):
    return str(v).strip().lower() not in ('', 'none', 'null', 'undefined', 'nan', 'no title given')

# --- Parse LLM output + apply fallbacks ---
rows = []
for row in todo.itertuples(index=False):
    raw = done.get((row.document_id, MODEL_ID))
    if raw is None:
        continue
    rec = {'document_id': row.document_id, 'file_name': row.file_name, 'parse_ok': True}
    try:
        data, _ = json.JSONDecoder().raw_decode(raw)
    except Exception:
        data, rec['parse_ok'] = {}, False
    source = squash(first_relevant_pages(row.pdf_path))
    for f in FIELDS:
        v = data.get(f)
        if f in LIST_FIELDS:
            rec[f] = '; '.join(as_list(v))
        else:
            rec[f] = '' if v is None else (clean_doi(v) if f == 'doi' else str(v).strip())

    # --- Observation fallback for the four scoreable fields ---
    for f in ['title', 'authors', 'doi', 'keywords']:
        obs_val  = str(OBS_MAP.get((row.document_id, f), '')).strip()
        pred_val = str(rec[f]).strip()

        if not is_valid(pred_val) and is_valid(obs_val):
            rec[f] = obs_val
            rec[f + '_source'] = 'observation'
        elif f == 'title' and is_valid(pred_val) and is_valid(obs_val):
            if len(squash(pred_val)) < len(squash(obs_val)) * 0.85:
                rec[f] = obs_val
                rec[f + '_source'] = 'observation_truncation_fix'
            else:
                rec[f + '_source'] = 'llm'
        elif f == 'authors' and is_valid(pred_val) and is_valid(obs_val):
            if len(as_list(pred_val)) < len(as_list(obs_val)) // 2:
                rec[f] = obs_val
                rec[f + '_source'] = 'observation_missing_authors'
            else:
                rec[f + '_source'] = 'llm'
        elif is_valid(pred_val):
            rec[f + '_source'] = 'llm'
        else:
            rec[f + '_source'] = 'missing'

    # --- PDF regex as last resort for DOI ---
    if not is_valid(rec['doi']):
        dois = find_dois_in_pdf(row.pdf_path)
        if dois:
            rec['doi'] = clean_doi(dois[0])
            rec['doi_source'] = 'regex_pdf'

    rec['grounded_title'] = bool(rec['title']) and squash(rec['title']) in source
    rec['grounded_doi'] = bool(rec['doi']) and squash(rec['doi']) in source
    names = as_list(rec['authors'])
    rec['grounded_authors'] = bool(names) and all(all(squash(t) in source for t in re.findall(r'\w{2,}', n)) for n in names)
    rec['doi_format_ok'] = (rec['doi'] == '') or bool(re.fullmatch(r'10\.\d{4,9}/[-._;()/:A-Z0-9]{4,}', rec['doi']))
    rec['type_ok'] = (rec['document_type'] == '') or rec['document_type'] in DOC_TYPES
    rows.append(rec)

extractions = pd.DataFrame(rows)
extractions.to_csv(OUT_DIR / 'eurollm_extractions.csv', index=False, encoding='utf-8-sig')
print('Parsed OK:', int(extractions.parse_ok.sum()), '/', len(extractions))
print('Filled fields:'); display((extractions[FIELDS] != '').sum().to_frame('documents'))
print('Grounded (of the documents where the field is filled):')
display(pd.DataFrame({f: [extractions.loc[extractions[f.replace('grounded_', '')] != '', f].mean()] for f in ['grounded_title', 'grounded_doi', 'grounded_authors']}).round(2))
display(extractions[['file_name', 'title', 'authors', 'doi', 'publication_year', 'document_type', 'language']].head(10))

# --- Field source breakdown ---
print('Field sources:')
for f in ['title', 'authors', 'doi', 'keywords']:
    col = f + '_source'
    if col in extractions.columns:
        print(f'\n{f}:')
        print(extractions[col].value_counts())

Parsed OK: 20 / 23
Filled fields:


,documents
title,20
authors,18
doi,7
publication_year,19
document_type,20
language,20
keywords,8


Grounded (of the documents where the field is filled):


,grounded_title,grounded_doi,grounded_authors
0,0.7,0.29,0.89


,file_name,title,authors,doi,publication_year,document_type,language
0,001_adaptive-capacities-for-diversified-flood-...,Adaptive Capacities for Diversified Flood Risk...,"Cosoveanu, Flavia Simona; Buijs, Jean-Marie; B...",null,2019,journal_article,en
1,005_building-relationships-in-higher-education...,Building relationships in higher education to ...,"Leenknecht, Martijn J.M.; Snijders, Ingrid; Wi...",null,2023,journal_article,en
2,011_fine-tuning-the-flavor-of-tetraselmis-chui...,,,,,,
3,013_grip-op-teksten-grip-op-kennis.pdf,"Grip op teksten, grip op kennis",Hilde Kooiker-den Boer; Ted Sanders; Jacquelin...,,2024,journal_article,en
4,016_impacts-of-counting-protocols-for-light-mi...,Impacts of counting protocols for light micros...,Pascalle Jacobs; Léon Serre-Fredj; Reinoud P.T...,null,2024,journal_article,en
5,019_lezen-en-schrijven-in-natuur-en-techniekon...,Lezen en schrijven in Natuur- en Techniekonder...,{'name': 'Hilde Kooiker-den Boer'}; {'name': '...,null,2025,journal_article,nl
6,023_relationship-quality-in-higher-education-a...,Relationship Quality in Higher Education and t...,Ingrid Snijders; Lisette Wijnia; Rebecca M. Ku...,10.1111/bjep.12455,2021,journal_article,en
7,027_the-use-of-tetraselmis-chuii-as-seafood-fl...,,,,,,
8,029_transition-towards-integrated-flood-risk-m...,Transition Towards Integrated Flood Risk Manag...,Jean-Marie Buijs; Flavia Simona Cosoveanu; Mar...,,2020,other,en
9,031_vakdidactische-kennisclips-over-tekststruc...,Vakdidactische kennisclips over tekststructuur...,H.S. Kooiker-den Boer; S.T.M. Bogaerds-Hazenbe...,null,2024,conference_paper,nl


## 6. Evaluate against the gold standard

Only gold cells that are filled in are scored. Title: similarity ≥ 0.90 counts as correct. Authors/keywords: set F1 (names compared as sorted tokens, so `Smith, J.` equals `J. Smith`). Everything else: exact match. Use `dev` while tuning; use `test` once at the end.

In [14]:

observations = pd.read_csv(
    INVENTORY_DIR / 'metadata_observations.csv',
    dtype=str, keep_default_na=False, encoding='utf-8-sig'
)

FIELD_MAP = {'title':'title', 'author':'authors', 'doi':'doi', 'pdf_keywords':'keywords'}
obs = observations[observations.field_name.isin(FIELD_MAP)].copy()
obs['field'] = obs.field_name.map(FIELD_MAP)

obs['priority'] = obs.source_type.map({'pdf_text': 0, 'pdf_embedded': 1}).fillna(2)
obs = obs.sort_values(['document_id', 'field', 'priority'])
obs = obs.drop_duplicates(['document_id', 'field'], keep='first')

truth = obs.pivot(index='document_id', columns='field', values='observed_value').reset_index()
for col in ['title', 'authors', 'doi', 'keywords']:
    if col not in truth.columns:
        truth[col] = ''
truth = truth.set_index('document_id')

pred = extractions.set_index('document_id')
common = truth.index.intersection(pred.index)

rows = []
for doc_id in common:
    g = truth.loc[doc_id]
    p = pred.loc[doc_id]
    for f in ['title', 'authors', 'doi', 'keywords']:
        obs_val = str(g[f]).strip()
        pred_val = str(p[f]).strip()
        if obs_val == '' and pred_val == '':
            continue
        rows.append({
            'document_id': doc_id,
            'file_name':   p['file_name'],
            'field':       f,
            'observation': obs_val,
            'prediction':  pred_val,
        })

comparison = pd.DataFrame(rows)
comparison.to_csv(OUT_DIR / 'comparison_observations.csv', index=False, encoding='utf-8-sig')

print(f'Documents compared: {len(common)}')
print(f'Rows in comparison: {len(comparison)}')
print()

with pd.option_context('display.max_colwidth', 120):
    for doc_id, group in comparison.groupby('document_id', sort=False):
        print('=' * 90)
        print(group['file_name'].iloc[0])
        print('-' * 90)
        for row in group.itertuples(index=False):
            print(f'{row.field:10s}  observation: {row.observation}')
            print(f'{"":10s}  prediction : {row.prediction}')
        print()

Documents compared: 20
Rows in comparison: 80

080_projectplan-leernetwerk-sport-bewegen-en-gezondheid-in-zeeland.pdf
------------------------------------------------------------------------------------------
title       observation: nan
            prediction : Projectplan Leernetwerk Sport, Bewegen en Gezondheid in Zeeland
authors     observation: Anne Muilenburg
            prediction : Anton Engels; Anne Muilenburg
doi         observation: nan
            prediction : 
keywords    observation: nan
            prediction : 

027_the-use-of-tetraselmis-chuii-as-seafood-flavoring-agent-in-a-vegetable-broth.pdf
------------------------------------------------------------------------------------------
title       observation: The use of Tetraselmis chuii as seafood flavoring agent in a vegetable broth
            prediction : 
authors     observation: Bert Coleman
            prediction : 
doi         observation: 10.1016/j.algal.2024.103538
            prediction : 
keywords    observa